# Notebook 03: Ingeniería de Características

## 1. Características adicionales

In [1]:
import os
import sys
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

sys.path.append(os.path.dirname(os.getcwd()))

from src import (
    read_csv_from_azure,
    COLUMNS,
    PATHS_PROJECT,
    TRANSFORMATION_DESCRIPTIONS,
    DataVersion,
    save_clean_dataset,
    create_version_changes,
    save_version_changes,
    get_signal_summary
)
from src.cloud_storage import upload_csv_to_azure
from src.constants import connection_string

In [2]:
df = read_csv_from_azure(connection_string, 'salva-health-data', 'pacientes_clean.csv')

signal_summaries = pd.DataFrame([
    get_signal_summary(patient_id, PATHS_PROJECT['signals']) for patient_id in df[COLUMNS['patient_id']]
])

df_features = df.merge(signal_summaries, on=COLUMNS['patient_id'])
df_features['imc'] = df_features[COLUMNS['weight']] / (df_features[COLUMNS['height']] / 100) ** 2
df_features.head(10)

,id_paciente,edad_paciente,sexo,peso_kg,altura_cm,fecha_registro,frecuencia_cardiaca_media_bpm,derivacion_ecg,frecuencia_muestreo_hz,etiqueta,n_samples,duration_seg,sampling_rate_hz,std_mV,range_mV,imc
0,P0305,82.0,M,69.8,168.0,2023-10-04,69.4,II,250,Normal,2500,9.996,250,0.1621,1.2189,24.730726
1,P0500,58.0,M,70.9,178.9,2023-04-23,79.2,II,250,Normal,2500,9.996,250,0.1685,1.2394,22.152643
2,P0442,49.0,M,84.2,173.1,2023-01-25,72.7,II,250,Normal,2500,9.996,250,0.1686,1.2379,28.100753
3,P0154,39.0,F,80.5,156.4,2023-06-24,87.0,II,250,Anormal,2500,9.996,250,0.1775,1.2779,32.909583
4,P0479,22.0,F,78.7,165.5,2023-01-28,77.8,II,250,Normal,2500,9.996,250,0.1736,1.2321,28.732852
5,P0132,34.0,M,75.5,170.9,2023-01-27,70.8,II,250,Normal,2500,9.996,250,0.1679,1.2450,25.850136
6,P0205,61.0,M,69.9,163.3,2023-06-29,89.1,II,250,Anormal,2500,9.996,250,0.1752,1.4055,26.212280
7,P0283,42.0,M,78.2,167.3,2023-11-02,64.1,II,250,Anormal,2500,9.996,250,0.1461,1.3246,27.939258
8,P0326,56.0,F,69.4,161.7,2023-02-21,74.2,II,250,Normal,2500,9.996,250,0.1627,1.2376,26.542354
9,P0248,26.0,F,67.3,169.5,2023-04-23,56.1,II,250,Normal,2500,9.996,250,0.1449,1.2382,23.424787


Se propone el Índice de Masa Corporal (IMC), calculado como peso_kg / (altura_cm/100)^2, por ser una medida clínica estándar que combina peso y altura en un solo indicador con interpretación médica directa, más informativo que ambas variables por separado para evaluar riesgo cardiovascular. Se evaluó también incorporar la frecuencia
dominante de la señal ECG vía FFT como feature adicional, pero se descartó debido que puede ser similar a la frecuencia cardiaca media y no aportar mucha información debido a los datos con que se cuenta. En su lugar, se considera std_mV ya calculada en el EDA a partir de la señal, dado que mostró una diferencia significativa entre clases con solapamiento razonable entre distribuciones.

## 2. Versionamiento de datos

In [3]:
df_features = df.copy()

feature_metadata = {
    'version': '3.0',
    'name': 'Patient Dataset with Features',
    'records': len(df_features),
    'columns': list(df_features.columns),
    'target_distribution': df_features[COLUMNS['label']].value_counts().to_dict(),
    'target_percentages': (df_features[COLUMNS['label']].value_counts(normalize=True) * 100).to_dict()
}

save_clean_dataset(df_features, DataVersion.FEATURES_FILE, feature_metadata)

transformations = [
    TRANSFORMATION_DESCRIPTIONS['feature_engineering'],
    TRANSFORMATION_DESCRIPTIONS['ecg_features']
]

changes = create_version_changes(
    len(df),
    len(df_features),
    transformations
)

changes['version_from'] = '2.0'
changes['version_to'] = '3.0'

save_version_changes(changes, PATHS_PROJECT['v3'])

print("\nVersioning complete.")


Versioning complete.


## 3. Subir a la nube

In [4]:
try:
    connection_string = os.environ['AZURE_CONNECTION_STRING']
    
    upload_csv_to_azure(
        df_features,
        connection_string,
        'salva-health-data',
        'pacientes_features.csv'
    )
    print("Dataset con características subido a Azure Storage exitosamente")
    
except Exception as e:
    print(f"Error al subir el dataset a Azure: {e}")

File uploaded to Azure: salva-health-data/pacientes_features.csv
Dataset con características subido a Azure Storage exitosamente
